In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np

In [3]:
#bag of words class, made during ml lab
class BagOfWords:
    def __init__(self):
        self.vocabulary = dict()
        self.words = []  
        
        
    def build_vocabulary(self, data):
        for sentence in data:
            for word in sentence:
                if word not in self.vocabulary:
                    self.vocabulary[word] = len(self.vocabulary)
                    self.words.append(word)
                    
            
    def get_features(self, data):
        features = np.zeros((len(data), len(self.vocabulary)))
        
        for id_sen, document in enumerate(data):
            for word in document:
                if word in self.vocabulary:
                    features[id_sen, self.vocabulary[word]] += 1
                    
        return features

In [4]:
#getting the data
def load_sample(file_name):
    f = open(file_name, 'r', encoding='utf8')
    
    indexes = []
    sentences = []
    
    for line in f.readlines():
        indexes.append(int("".join(line[:6])))
        sentences.append(line[7:].strip('\n').split())
        
    return indexes, sentences


def load_label(file_name):
    f = open(file_name, 'r', encoding='utf8')
    
    sentences = []
    
    for line in f.readlines():
        sentences.append(int(line[7]))
        
    return sentences

In [5]:
#train data
train_indexes, train_samples = load_sample("data/train_samples.txt")
train_labels = load_label("data/train_labels.txt")

#validation data
validation_indexes, validation_samples = load_sample("data/validation_samples.txt")
validation_labels = load_label("data/validation_labels.txt")

#test data
test_indexes, test_samples = load_sample("data/test_samples.txt")

In [5]:
bow = BagOfWords()
bow.build_vocabulary(train_samples)

train_features = bow.get_features(train_samples)
validation_features = bow.get_features(validation_samples)

In [6]:
#train the model
gradient_model = GradientBoostingClassifier(n_estimators=20, learning_rate=0.5)
gradient_model.fit(train_features, train_labels)

GradientBoostingClassifier(learning_rate=0.5, n_estimators=20)

In [7]:
predicted = gradient_model.predict(validation_features)

print(np.mean(predicted == validation_labels))

0.6424


In [8]:
bow_final = BagOfWords()
bow_final.build_vocabulary(train_samples + validation_samples)

all_train_features = bow_final.get_features(train_samples + validation_samples)
test_features = bow_final.get_features(test_samples)

#train the model
gradient_model_final = GradientBoostingClassifier(n_estimators=20, learning_rate=0.5)
gradient_model_final.fit(all_train_features, train_labels + validation_labels)

#get the prediction on the test data
predicted_final = gradient_model_final.predict(test_features)

#and write it in the csv
g = open("data/test_labels.txt", 'w')
g.write("id,label\n")

for idx in range(len(predicted_final)):
    g.write(f"{test_indexes[idx]},{predicted_final[idx]}\n")